# Day 25 – ShapeNet: Geometry-Guided Lightweight Neural Networks for STEM Images

**Both challenges solved:**

| Challenge | Object geometry used | Network | Output |
|---|---|---|---|
| **1. SiSb crystalline / amorphous boundary** | Crystal = sharp **Bragg spots** at the measured lattice wave-vectors; amorphous = the same spacing but spread over a **ring** (no orientational order) | **BraggShapeNet** (~8 k parameters): learnable Fourier band-pass filters centred on the measured wave-vectors + ring filter, then a 4-layer dilated CNN | Crystal probability map, boundary contours (CSV), crystal fraction vs. frame |
| **2. YBCO atomic columns** | Columns = compact 2-D Gaussian blobs on a lattice with a **known nearest-neighbour distance (≈2.86 Å)** → every image is rescaled so columns are always 10 px apart | **AtomUNet** (~90 k parameters) trained only on physics-based synthetic images, then sub-pixel peak refinement | Atomic coordinates (px and nm, CSV) + overlays |

Everything runs end-to-end in Google Colab: **Runtime → Run all**. A GPU runtime takes about 3–5 min; CPU also works, just slower.

**Pipeline idea (the "ShapeNet" philosophy).** We measure the object's geometry first (FFT or autocorrelation), then hard-wire it into the network: the filter positions for SiSb and the input scale for YBCO. Because the geometry is fixed, the trainable part can be tiny and trained in a few hundred steps, and it transfers across magnifications.

## 0. Setup (installs, data download, imports)

In [ ]:
import sys, subprocess, os, importlib.util

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for mod, pkg in [("gdown", "gdown"), ("h5py", "h5py"), ("skimage", "scikit-image"),
                 ("sklearn", "scikit-learn"), ("pandas", "pandas")]:
    if importlib.util.find_spec(mod) is None:
        _pip(pkg)

from pathlib import Path
DATA_ROOT = Path(os.environ.get("DAY25_DATA", "day_25 data"))
DRIVE_URL = "https://drive.google.com/drive/folders/1h2v6YRPG7cwi7kuMPPPdwV0P0PYu49oH?usp=sharing"
if not (DATA_ROOT / "SiSb").exists():
    import gdown
    gdown.download_folder(url=DRIVE_URL, quiet=True, use_cookies=False)
print("Data folder:", DATA_ROOT.resolve(), "| exists:", DATA_ROOT.exists())

In [ ]:
import re, json, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu
from skimage.registration import phase_cross_correlation
from skimage import measure, morphology
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FAST = DEVICE == "cpu"            # fewer training steps on CPU
OUT = Path("day25_results"); OUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "image.cmap": "gray", "axes.titlesize": 9})
print("torch", torch.__version__, "| device:", DEVICE, "| fast mode:", FAST)
T0 = time.time()

## 1. Data loading
Velox `.emd` files are HDF5. We read them directly with `h5py` (fast, no extra dependency) and also pull out the **pixel size** from the metadata, because the YBCO geometry prior needs it. If `h5py` cannot read a file, the code falls back to `pyTEMlib`.

In [ ]:
def natural_sort_key(path: Path):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", path.name)]

def list_emd_files(folder) -> list:
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder}")
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".emd"],
                  key=natural_sort_key)

def read_emd(path):
    """Return (image float32, pixel size in Angstrom or None, detector name)."""
    try:
        import h5py
        best = None
        with h5py.File(path, "r") as h:
            grp = h["Data/Image"]
            for key in grp:
                d = grp[key]
                try:
                    meta = json.loads(bytes(np.asarray(d["Metadata"][:, 0])).decode("utf-8", "ignore").rstrip("\x00"))
                except Exception:
                    meta = {}
                det = str(meta.get("BinaryResult", {}).get("Detector", ""))
                arr = np.asarray(d["Data"], dtype=np.float32)
                if arr.ndim == 3:
                    arr = arr.mean(-1)
                px = meta.get("BinaryResult", {}).get("PixelSize", {}).get("width")
                px_A = float(px) * 1e10 if px is not None else None
                cand = (arr, px_A, det)
                if "HAADF" in det.upper():
                    return cand
                best = best or cand
        if best is not None:
            return best
    except Exception as e:
        print("h5py failed on", Path(path).name, "->", e, "; trying pyTEMlib")
    _pip("pyTEMlib")
    import pyTEMlib.file_tools as ft
    ds = ft.open_file(str(path))
    ds = ds if isinstance(ds, dict) else {"0": ds}
    for v in ds.values():
        arr = np.squeeze(np.asarray(v)).astype(np.float32)
        if arr.ndim == 2:
            return arr, None, "unknown"
    raise ValueError(f"No 2D image in {path}")

def load_folder(folder, n=None):
    files = list_emd_files(folder)[:n]
    out = []
    for f in files:
        img, px, det = read_emd(f)
        out.append(dict(name=f.stem, img=img, px_A=px, det=det))
    return out

sisb = load_folder(DATA_ROOT / "SiSb", n=12)
ybco = load_folder(DATA_ROOT / "YBCO")
print(f"SiSb: {len(sisb)} images | YBCO: {len(ybco)} images")
print(pd.DataFrame([dict(set=s, name=d["name"][:40], shape=d["img"].shape,
                         pixel_pm=round(d["px_A"] * 100, 2) if d["px_A"] else None)
                    for s, L in [("SiSb", sisb), ("YBCO", ybco)] for d in L]).to_string())

### Shared helpers (display and differentiable Gaussian blur)

In [ ]:
def show(ax, img, title=None, p=(0.5, 99.5), **kw):
    lo, hi = np.percentile(img, p)
    ax.imshow(img, vmin=lo, vmax=hi, **kw); ax.axis("off")
    if title: ax.set_title(title)

def _gk1d(sigma, device):
    r = max(1, int(math.ceil(3 * sigma)))
    x = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    k = torch.exp(-x ** 2 / (2 * sigma ** 2)); return k / k.sum()

def gblur(x, sigma):
    """Separable Gaussian blur of a (B,C,H,W) tensor."""
    k = _gk1d(sigma, x.device); r = len(k) // 2; C = x.shape[1]
    mode = "reflect" if r < min(x.shape[-2:]) else "replicate"
    x = F.conv2d(F.pad(x, (r, r, 0, 0), mode=mode), k.view(1, 1, 1, -1).repeat(C, 1, 1, 1), groups=C)
    x = F.conv2d(F.pad(x, (0, 0, r, r), mode=mode), k.view(1, 1, -1, 1).repeat(C, 1, 1, 1), groups=C)
    return x

def local_norm(x, sigma):
    """Remove slowly varying background and local contrast changes (thickness, dark bands)."""
    m = gblur(x, sigma); v = gblur((x - m) ** 2, sigma)
    return ((x - m) / torch.sqrt(v + 1e-6)).clamp(-5, 5)

def t2(img):
    return torch.as_tensor(np.ascontiguousarray(img), dtype=torch.float32)[None, None]

---
# CHALLENGE 1 — Crystalline / amorphous boundary in SiSb

## 1.1 Explore the geometry
What distinguishes the two phases:
* **Crystal:** periodic lattice, so its local power spectrum has a few sharp **Bragg spots** at |k| = 1/d_hkl in specific *directions*.
* **Amorphous:** similar interatomic distances but no orientational order, so the power at the same |k| is spread uniformly around a **ring**.
* **HAADF intensity does not separate them reliably.** The dark loop in frames 03–12 is a thickness or composition contrast that does not follow the phase boundary (shown below).
* **More than one grain orientation:** below the loop in frames 05–12 a second, rotated (twinned) grain appears. A detector tuned to one orientation would miss it.

So the right *shape descriptor* is **rotation-invariant**: at the measured lattice radius |k|, how peaked is the local angular power distribution? We use **max over angle ÷ mean over angle**. It is large for any crystal grain, whatever its orientation, and about 1 for amorphous material.

The measured spacings are **3.2 Å and 1.98 Å**, with 70.5° between the two {111} reflections. These are the Si(111) and Si(220) planes viewed down [110], a built-in sanity check of the geometry step.

In [ ]:
N_ANG = 18                    # angular filters over [0, pi)

def find_bragg_vectors(img, n_unique=3, k_min=0.03, k_max=0.35, nms=15):
    """Strongest lattice wave-vectors (cycles/px), one per +/- pair, from the windowed power spectrum."""
    im = img - img.mean(); H, W = im.shape
    win = np.outer(np.hanning(H), np.hanning(W))
    P = ndi.gaussian_filter(np.log1p(np.fft.fftshift(np.abs(np.fft.fft2(im * win)) ** 2)), 1.5)
    FX, FY = np.meshgrid(np.fft.fftshift(np.fft.fftfreq(W)), np.fft.fftshift(np.fft.fftfreq(H)))
    R = np.hypot(FX, FY); rb = np.round(R * min(H, W)).astype(int)
    radial = np.bincount(rb.ravel(), P.ravel()) / np.maximum(np.bincount(rb.ravel()), 1)
    Q = P - radial[rb]                                   # remove isotropic (ring) background
    Q[(R < k_min) | (R > k_max) | ~((FX > 0) | ((FX == 0) & (FY > 0)))] = -np.inf
    pk = (Q == ndi.maximum_filter(Q, nms)) & np.isfinite(Q) & (Q > 0)
    ys, xs = np.nonzero(pk); o = np.argsort(-Q[ys, xs])[:n_unique]
    return np.c_[FX[ys[o], xs[o]], FY[ys[o], xs[o]]].astype(np.float32), P

def lattice_radii(k, n=2):
    """Distinct |k| values (lattice-plane families), padded to n by repetition."""
    kn = np.sort(np.hypot(k[:, 0], k[:, 1]))
    r = [kn[0]] + [v for a, v in zip(kn[:-1], kn[1:]) if v > a * 1.05]
    return np.array((r * n)[:n], np.float32)

def angular_energy(x, radii, rbw, abw, n_ang=N_ANG, pool=4, blur=3.0):
    """Local band-pass energy around |k| = radius, resolved in n_ang directions.
    x (B,1,H,W); radii (B,R) cycles/px; rbw (R,) radial width / |k|; abw angular width (rad).
    Returns (B, R, n_ang, H/pool, W/pool)."""
    B, _, H, W = x.shape; dev = x.device
    FY, FX = torch.meshgrid(torch.fft.fftfreq(H, device=dev), torch.fft.fftfreq(W, device=dev), indexing="ij")
    Rf = torch.sqrt(FX ** 2 + FY ** 2); TH = torch.atan2(FY, FX)
    X = torch.fft.fft2(x[:, 0])
    out = []
    for i in range(radii.shape[1]):
        r = radii[:, i, None, None]
        radial = torch.exp(-(Rf - r) ** 2 / (2 * (rbw[i] * r) ** 2))
        per = []
        for j in range(n_ang):
            dth = torch.remainder(TH - math.pi * j / n_ang + math.pi, 2 * math.pi) - math.pi
            G = radial * torch.exp(-dth ** 2 / (2 * abw ** 2))          # one-sided -> envelope amplitude
            per.append(F.avg_pool2d((torch.fft.ifft2(X * G).abs() ** 2)[:, None], pool))
        out.append(gblur(torch.cat(per, 1), blur))
    return torch.stack(out, 1)

def order_features(E):
    """Rotation-invariant features per lattice radius: log max_theta, log mean_theta, and their ratio."""
    mx = torch.log(E.amax(2) + 1e-4); mn = torch.log(E.mean(2) + 1e-4)
    return torch.cat([mx, mn, mx - mn], 1)

SISB_NORM_SIGMA = 40.0

@torch.no_grad()
def classical_order_map(img, radii, pad=64):
    """Non-learned baseline: mean over radii of log(max_theta E / mean_theta E), full resolution."""
    x = F.pad(local_norm(t2(img).to(DEVICE), SISB_NORM_SIGMA), (pad,) * 4, mode="reflect")
    R = len(radii)
    E = angular_energy(x, torch.as_tensor(radii)[None].to(DEVICE), torch.full((R,), 0.15, device=DEVICE), math.pi / N_ANG)
    f = order_features(E)[:, 2 * R:].mean(1, keepdim=True)
    return F.interpolate(f, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0, pad:-pad, pad:-pad].cpu().numpy()

for d in sisb:
    d["k"], d["P"] = find_bragg_vectors(d["img"])
    d["radii"] = lattice_radii(d["k"])
    d["d_lat"] = float(1 / d["radii"].min())                   # largest lattice period (px)
    d["order"] = classical_order_map(d["img"], d["radii"])

print(pd.DataFrame([dict(frame=d["name"][:2], pixel_pm=round(d["px_A"] * 100, 2),
                         lattice_spacings_px=[round(float(v), 2) for v in 1 / np.hypot(d["k"][:, 0], d["k"][:, 1])],
                         lattice_spacings_A=[round(float(v), 2) for v in d["px_A"] / np.hypot(d["k"][:, 0], d["k"][:, 1])],
                         angles_deg=[round(float(v), 1) for v in np.degrees(np.arctan2(d["k"][:, 1], d["k"][:, 0]))])
                    for d in sisb]).to_string())

In [ ]:
d = sisb[9]; H, W = d["img"].shape
fig, ax = plt.subplots(1, 5, figsize=(21, 4.5))
show(ax[0], d["img"], f"SiSb {d['name'][:2]} HAADF")
c = H // 2
ax[1].imshow(d["P"][c - 300:c + 300, c - 300:c + 300], cmap="magma"); ax[1].axis("off")
for kx, ky in d["k"]:
    for s in (1, -1):
        ax[1].plot(300 + s * kx * W, 300 + s * ky * H, "co", mfc="none", ms=12)
ax[1].set_title("Power spectrum: Bragg spots (circles) + amorphous ring")
show(ax[2], ndi.gaussian_filter(d["img"], 2 * d["d_lat"]), "Smoothed HAADF intensity (NOT the phase)")
ax[3].imshow(d["order"], cmap="magma"); ax[3].axis("off"); ax[3].set_title("Rotation-invariant order (max/mean over angle)")
pc = d["img"][100:356, 900:1156]; pa = d["img"][1750:2006, 1500:1756]
lf = lambda p: np.log1p(np.abs(np.fft.fftshift(np.fft.fft2((p - p.mean()) * np.outer(np.hanning(256), np.hanning(256))))) ** 2)
ax[4].imshow(np.hstack([ndi.gaussian_filter(lf(pc), 1), ndi.gaussian_filter(lf(pa), 1)])[48:208], cmap="magma")
ax[4].axis("off"); ax[4].set_title("Local FFT: crystal patch (spots) | amorphous patch (ring)")
plt.tight_layout(); plt.savefig(OUT / "sisb_geometry_exploration.png", dpi=110); plt.show()

## 1.2 BraggShapeNet: a geometry-guided lightweight network
* **Layer 1 (physics layer, 5 learnable parameters):** a bank of Fourier-space filters at the lattice radii |k| measured from *this* image's FFT, in 18 directions. The radial width, angular width and a small radius correction are learned. The local energy in each (radius, direction) channel measures how much lattice of that orientation is present.
* **Rotation-invariant pooling:** for each radius we keep log max over angle, log mean over angle and their ratio. Any grain orientation (including the twin) gives the same response, so the network generalises across grains, orientations and magnifications.
* **Head:** 4 dilated 3×3 convolutions (receptive field about 125 px, about 9 lattice periods), then a 1×1 convolution to a crystal logit. Bilinear upsampling gives a full-resolution probability map. Total about 8 k parameters.
* **Training data (no manual labels):**
  1. **Physics-based synthetic images** with an exact ground-truth mask. They contain one or two randomly oriented crystal grains (lattice texture with strain noise) and amorphous regions (random-phase noise restricted to the same |k| rings). Boundaries are random, intensity variations are deliberately *uncorrelated* with the phase, and Poisson noise is added at random dose.
  2. **Real frames** with *confident* pseudo-labels from the classical order map: only pixels > 4 lattice periods from the Otsu boundary are used, and the rest are ignored. Frames **04, 08 and 12 are held out** for evaluation.

In [ ]:
class BraggShapeNet(nn.Module):
    def __init__(self, n_r=2, n_ang=N_ANG, hidden=16, pool=4):
        super().__init__()
        self.n_ang, self.pool = n_ang, pool
        self.log_rbw = nn.Parameter(torch.full((n_r,), math.log(0.15)))   # radial width / |k|
        self.log_abw = nn.Parameter(torch.tensor(math.log(math.pi / n_ang)))  # angular width (rad)
        self.rcorr = nn.Parameter(torch.zeros(n_r))                         # radius correction (<= +-10 %)
        layers, c = [], 3 * n_r
        for dil in (1, 2, 4, 8):
            layers += [nn.Conv2d(c, hidden, 3, padding=dil, dilation=dil), nn.ReLU(inplace=True)]; c = hidden
        layers += [nn.Conv2d(hidden, 1, 1)]
        self.head = nn.Sequential(*layers)

    def features(self, x, radii):
        r = radii * (1 + 0.1 * torch.tanh(self.rcorr))[None]
        E = angular_energy(x, r, torch.exp(self.log_rbw), torch.exp(self.log_abw), self.n_ang, self.pool)
        return order_features(E)

    def forward(self, x, radii, pad=32):
        H, W = x.shape[-2:]
        xp = F.pad(x, (pad, pad, pad, pad), mode="reflect")           # no FFT wrap-around at the edges
        logit = self.head(self.features(xp, radii))
        logit = F.interpolate(logit, size=xp.shape[-2:], mode="bilinear", align_corners=False)
        return logit[..., pad:pad + H, pad:pad + W]

sisb_net = BraggShapeNet().to(DEVICE)
print("BraggShapeNet trainable parameters:", sum(p.numel() for p in sisb_net.parameters()))

In [ ]:
def _lattice_texture(k, xx, yy, rng, H, W):
    strain = ndi.gaussian_filter(rng.standard_normal((H, W)), 30) * rng.uniform(0, 60)
    c = sum(np.cos(2 * np.pi * (kx * xx + ky * yy) + rng.uniform(0, 2 * np.pi) + strain) for kx, ky in k)
    t = np.exp(c * rng.uniform(0.4, 1.0)); return (t - t.mean()) / (t.std() + 1e-6)

def _rot(k, th):
    return k @ np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]).T

def synth_sisb(size, k_ref, rng):
    """Synthetic crystal/amorphous image + exact mask + measured lattice radii."""
    H = W = size
    k = _rot(k_ref, rng.uniform(0, np.pi)) * rng.uniform(0.9, 1.1)
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    t = _lattice_texture(k, xx, yy, rng, H, W)
    if rng.random() < 0.5:                                           # second, rotated grain (e.g. twin)
        t2_ = _lattice_texture(_rot(k, rng.uniform(0.3, np.pi - 0.3)), xx, yy, rng, H, W)
        g = ndi.gaussian_filter(rng.standard_normal((H, W)), 30) > 0
        t = np.where(g, t, t2_)
    FX, FY = np.meshgrid(np.fft.fftfreq(W), np.fft.fftfreq(H)); R = np.hypot(FX, FY)
    kn = np.hypot(k[:, 0], k[:, 1])
    ring = np.max([np.exp(-(R - kk) ** 2 / (2 * (rng.uniform(0.1, 0.22) * kk) ** 2)) for kk in kn], 0)
    a = np.real(np.fft.ifft2(np.fft.fft2(rng.standard_normal((H, W))) * ring)); a /= a.std() + 1e-6
    mode = rng.choice(4, p=[0.5, 0.3, 0.1, 0.1])
    if mode == 2: M = np.ones((H, W))
    elif mode == 3: M = np.zeros((H, W))
    else:
        f = ndi.gaussian_filter(rng.standard_normal((H, W)), rng.uniform(12, 40)); f /= f.std() + 1e-6
        phi = rng.uniform(0, 2 * np.pi)
        ramp = ((xx - W / 2) * np.cos(phi) + (yy - H / 2) * np.sin(phi)) / H
        f = f * (0.4 if mode == 1 else 1.0) + ramp * (rng.uniform(2, 6) if mode == 1 else rng.uniform(0, 2))
        M = (f > rng.uniform(-0.5, 0.5)).astype(np.float32)
    Ms = ndi.gaussian_filter(M.astype(np.float32), rng.uniform(0.5, 3))
    tex = Ms * t + (1 - Ms) * a * rng.uniform(0.6, 1.3)
    base = 1 + 2.0 * ndi.gaussian_filter(rng.standard_normal((H, W)), 40)
    base += rng.uniform(-0.2, 0.2) * Ms                                # intensity NOT tied to phase
    blob = ndi.gaussian_filter((rng.random((H, W)) < 2e-4).astype(float), 25)
    base -= rng.uniform(0, 0.3) * blob / (blob.max() + 1e-6)          # dark patches (like the real loop)
    img = np.clip(base, 0.2, None) * (1 + rng.uniform(0.08, 0.35) * tex)
    img = ndi.gaussian_filter(img, rng.uniform(0.3, 1.0))
    dose = np.exp(rng.uniform(np.log(3), np.log(80)))
    img = rng.poisson(np.clip(img, 0, None) * dose).astype(np.float32) / dose
    radii = lattice_radii(k) * (1 + rng.normal(0, 0.01, 2))           # emulate measurement error
    return img.astype(np.float32), (M > 0.5).astype(np.float32), radii.astype(np.float32)

# ---- pseudo-labels for real frames
HOLDOUT = [3, 7, 11]          # frames 04, 08, 12
for d in sisb:
    o = d["order"]; thr = threshold_otsu(o)
    m = morphology.remove_small_objects(o > thr, 20000)
    m = ~morphology.remove_small_objects(~m, 20000)
    d["mask_classic"] = m
    dist = ndi.distance_transform_edt(m) + ndi.distance_transform_edt(~m)
    lab = m.astype(np.float32); lab[dist < 4 * d["d_lat"]] = -1                   # ignore band
    d["pseudo"] = lab
    d["x_norm"] = local_norm(t2(d["img"]), SISB_NORM_SIGMA)[0, 0]

rng = np.random.default_rng(1)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    im_, m_, _ = synth_sisb(512, sisb[6]["k"], rng)
    show(ax[0, i], im_, "synthetic"); ax[1, i].imshow(m_, cmap="viridis"); ax[1, i].axis("off"); ax[1, i].set_title("true mask (yellow = crystal)")
plt.tight_layout(); plt.show()

In [ ]:
def train_sisb(model, n_iter, crop=256, b_syn=4, b_real=4, lr=3e-3, seed=0):
    rng = np.random.default_rng(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n_iter)
    train_ids = [i for i in range(len(sisb)) if i not in HOLDOUT]
    hist = []
    for it in range(n_iter):
        xs, ys, rs = [], [], []
        for _ in range(b_syn):
            im_, m_, r_ = synth_sisb(crop, sisb[rng.choice(train_ids)]["k"], rng)
            xs.append(local_norm(t2(im_), SISB_NORM_SIGMA)[0]); ys.append(torch.from_numpy(m_)[None]); rs.append(torch.from_numpy(r_))
        for _ in range(b_real):
            d = sisb[rng.choice(train_ids)]; H, W = d["img"].shape
            y0, x0 = rng.integers(0, H - crop), rng.integers(0, W - crop)
            xc = d["x_norm"][y0:y0 + crop, x0:x0 + crop]; lc = d["pseudo"][y0:y0 + crop, x0:x0 + crop]
            if rng.random() < 0.5: xc, lc = xc.flip(-1), lc[:, ::-1]
            xs.append(xc[None]); ys.append(torch.from_numpy(np.ascontiguousarray(lc))[None]); rs.append(torch.from_numpy(d["radii"]))
        x = torch.stack(xs).to(DEVICE); y = torch.stack(ys).to(DEVICE); r = torch.stack(rs).to(DEVICE)
        logit = model(x, r)
        valid = (y >= 0).float()
        loss = (F.binary_cross_entropy_with_logits(logit, y.clamp(0, 1), reduction="none") * valid).sum() / valid.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        hist.append(loss.item())
        if it % 100 == 0 or it == n_iter - 1:
            print(f"iter {it:4d}  loss {np.mean(hist[-50:]):.4f}")
    return hist

t = time.time()
sisb_hist = train_sisb(sisb_net, n_iter=300 if FAST else 600)
print(f"SiSb training time: {time.time() - t:.1f}s")
print("learned radial width/|k|:", torch.exp(sisb_net.log_rbw).detach().cpu().numpy().round(3),
      "| angular width (deg):", round(math.degrees(float(torch.exp(sisb_net.log_abw))), 1),
      "| radius correction:", (0.1 * torch.tanh(sisb_net.rcorr)).detach().cpu().numpy().round(3))

## 1.3 Evaluation
**(a) Synthetic test set with exact ground truth** (20 unseen 512×512 images). We compare three methods:
* HAADF intensity + Otsu (naive baseline; we even let it pick the better polarity),
* the classical rotation-invariant order map + Otsu (no learning),
* BraggShapeNet.

**(b) Real held-out frames 04, 08 and 12:** agreement between the network and the classical map. There is no human ground truth for these.

In [ ]:
@torch.no_grad()
def predict_sisb(model, img, radii):
    model.eval()
    x = local_norm(t2(img).to(DEVICE), SISB_NORM_SIGMA)
    p = torch.sigmoid(model(x, torch.as_tensor(radii)[None].to(DEVICE), pad=64))[0, 0].cpu().numpy()
    model.train(); return p

def clean_mask(m, min_area):
    m = morphology.remove_small_objects(m.astype(bool), min_area)
    return ~morphology.remove_small_objects(~m, min_area)

def iou(a, b):
    u = (a | b).sum()
    return 1.0 if u == 0 else (a & b).sum() / u

def miou(a, b):
    """Mean IoU over the two phases (crystal and amorphous)."""
    return 0.5 * (iou(a, b) + iou(~a, ~b))

def boundary_dist(a, b):
    """Symmetric mean distance between the two phase boundaries (px)."""
    ba = a ^ ndi.binary_erosion(a, border_value=1); bb = b ^ ndi.binary_erosion(b, border_value=1)
    if not ba.any() and not bb.any(): return 0.0
    if not ba.any() or not bb.any(): return np.nan
    return 0.5 * (ndi.distance_transform_edt(~bb)[ba].mean() + ndi.distance_transform_edt(~ba)[bb].mean())

rng = np.random.default_rng(123); rows = []
for i in range(20):
    ref = sisb[HOLDOUT[i % 3]]
    im_, gt, r_ = synth_sisb(512, ref["k"], rng); gt = gt > 0.5
    dl = 1 / r_.min(); amin = int((3 * dl) ** 2)
    sm = ndi.gaussian_filter(im_, 2 * dl)
    m_int = sm > threshold_otsu(sm); m_int = m_int if iou(m_int, gt) >= iou(~m_int, gt) else ~m_int
    o = classical_order_map(im_, r_); m_cls = clean_mask(o > threshold_otsu(o), amin)
    m_nn = clean_mask(predict_sisb(sisb_net, im_, r_) > 0.5, amin)
    for name, m in [("HAADF intensity + Otsu", m_int), ("Classical order map + Otsu (no learning)", m_cls), ("BraggShapeNet (ours)", m_nn)]:
        rows.append(dict(method=name, mIoU=miou(m, gt), pixel_acc=(m == gt).mean(), boundary_err_px=boundary_dist(m, gt)))
syn_sisb = pd.DataFrame(rows).groupby("method", sort=False).agg(["mean", "std"]).round(3)
print("Synthetic test set (20 images, exact ground truth):"); print(syn_sisb.to_string())
syn_sisb.to_csv(OUT / "sisb_synthetic_benchmark.csv")

In [ ]:
for d in sisb:
    amin = int((3 * d["d_lat"]) ** 2)
    d["prob"] = predict_sisb(sisb_net, d["img"], d["radii"])
    d["mask_nn"] = clean_mask(d["prob"] > 0.5, 20000)

rows = []
for i in HOLDOUT:
    d = sisb[i]; conf = d["pseudo"] >= 0
    rows.append(dict(frame=d["name"][:2], mIoU_vs_classical=miou(d["mask_nn"], d["mask_classic"]),
                     acc_on_confident_pixels=(d["mask_nn"][conf] == (d["pseudo"][conf] > 0.5)).mean(),
                     boundary_diff_nm=boundary_dist(d["mask_nn"], d["mask_classic"]) * d["px_A"] / 10))
print("Real held-out frames (never seen in training):"); print(pd.DataFrame(rows).round(3).to_string())

## 1.4 Boundary visualisation and physical read-out

In [ ]:
def contours_of(mask, smooth=2.0):
    return measure.find_contours(ndi.gaussian_filter(mask.astype(float), smooth), 0.5)

(OUT / "sisb_boundaries").mkdir(exist_ok=True)
fig, ax = plt.subplots(3, 4, figsize=(18, 13.5)); ax = ax.ravel()
for i, d in enumerate(sisb):
    show(ax[i], d["img"], f"{d['name'][:2]}  crystal fraction {d['mask_nn'].mean():.2f}")
    ax[i].imshow(np.ma.masked_where(~d["mask_nn"], d["mask_nn"]), cmap="autumn", alpha=0.18)
    rec = []
    for j, cnt in enumerate(contours_of(d["mask_nn"])):
        ax[i].plot(cnt[:, 1], cnt[:, 0], "c-", lw=1.6)
        rec.append(pd.DataFrame(dict(contour_id=j, y_px=cnt[:, 0], x_px=cnt[:, 1],
                                     y_nm=cnt[:, 0] * d["px_A"] / 10, x_nm=cnt[:, 1] * d["px_A"] / 10)))
    if rec: pd.concat(rec).to_csv(OUT / "sisb_boundaries" / f"{d['name'][:2]}_boundary.csv", index=False)
    if i in HOLDOUT: ax[i].set_title(ax[i].get_title() + "  [held-out]", color="red")
plt.suptitle("Challenge 1 – detected crystalline (orange tint) / amorphous boundary (cyan)", fontsize=14)
plt.tight_layout(); plt.savefig(OUT / "sisb_boundaries_all_frames.png", dpi=100); plt.show()

In [ ]:
d = sisb[9]
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
show(ax[0], d["img"], f"Frame {d['name'][:2]} (full resolution)")
for cnt in contours_of(d["mask_nn"]): ax[0].plot(cnt[:, 1], cnt[:, 0], "c-", lw=1.5)
ax[1].imshow(d["prob"], cmap="RdYlBu_r", vmin=0, vmax=1); ax[1].axis("off"); ax[1].set_title("BraggShapeNet crystal probability")
y0, x0 = 1000, 500
show(ax[2], d["img"][y0:y0 + 400, x0:x0 + 400], "Zoom: lattice fringes stop at the boundary")
for cnt in contours_of(d["mask_nn"]):
    ax[2].plot(cnt[:, 1] - x0, cnt[:, 0] - y0, "c-", lw=2)
ax[2].set_xlim(0, 400); ax[2].set_ylim(400, 0)
plt.tight_layout(); plt.savefig(OUT / "sisb_zoom_frame10.png", dpi=110); plt.show()

In [ ]:
# Time series (frames 03-12 share magnification): drift-correct and overlay the boundaries
same = [i for i, d in enumerate(sisb) if abs(d["px_A"] - sisb[-1]["px_A"]) < 1e-3]
ref_s = ndi.gaussian_filter(sisb[same[-1]]["img"], 8)[::4, ::4]
shifts = {i: phase_cross_correlation(ref_s, ndi.gaussian_filter(sisb[i]["img"], 8)[::4, ::4], upsample_factor=4)[0] * 4 for i in same}
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
show(ax[0], sisb[same[-1]]["img"], "Boundary evolution (drift-corrected), frames " + ", ".join(sisb[i]["name"][:2] for i in same))
cols = plt.cm.viridis(np.linspace(0, 1, len(same)))
for c_, i in zip(cols, same):
    for cnt in contours_of(sisb[i]["mask_nn"]):
        ax[0].plot(cnt[:, 1] + shifts[i][1], cnt[:, 0] + shifts[i][0], "-", color=c_, lw=1.3)
frac = [sisb[i]["mask_nn"].mean() for i in range(len(sisb))]
area_nm2 = [sisb[i]["mask_nn"].sum() * (sisb[i]["px_A"] / 10) ** 2 for i in range(len(sisb))]
blen = [sum(np.sum(np.hypot(*np.diff(c, axis=0).T)) for c in contours_of(sisb[i]["mask_nn"])) * sisb[i]["px_A"] / 10 for i in range(len(sisb))]
ax[1].plot(range(1, 13), frac, "o-", label="crystal area fraction")
ax[1].set_xlabel("frame"); ax[1].set_ylabel("crystalline fraction of field of view")
ax2 = ax[1].twinx(); ax2.plot(range(1, 13), blen, "s--", color="C3", label="boundary length (nm)"); ax2.set_ylabel("boundary length (nm)")
ax[1].axvspan(0.5, 2.5, color="0.9"); ax[1].text(1.5, min(frac), "different\nmagnification", ha="center", fontsize=8)
ax[1].legend(loc="upper left"); ax2.legend(loc="upper right"); ax[1].set_title("Physical read-out per frame")
plt.tight_layout(); plt.savefig(OUT / "sisb_time_evolution.png", dpi=110); plt.show()
sisb_summary = pd.DataFrame(dict(frame=[d["name"] for d in sisb], crystal_fraction=np.round(frac, 4),
                                 crystal_area_nm2=np.round(area_nm2, 1), boundary_length_nm=np.round(blen, 1)))
sisb_summary.to_csv(OUT / "sisb_summary.csv", index=False); print(sisb_summary.to_string())

---
# CHALLENGE 2 — Atomic-column finding in YBCO

## 2.1 Explore the geometry
* Columns are compact, roughly Gaussian blobs (some are elongated by scan distortion) on a 2-D lattice. Bright Ba/Y rows alternate with dimmer Cu–O rows, and dark bands run along the layers.
* The **autocorrelation function (ACF)** gives the nearest-neighbour column spacing. Combined with the pixel size, **every image gives about 2.86 Å** (the centred-square cation sub-lattice). Images where the centred columns are weak show 4.05 Å instead.
* **Geometry prior:** rescale every image so that the 2.86 Å spacing becomes **exactly 10 px**. In that frame atoms always have the same size and spacing, so one small network serves all magnifications (1000 kx – 4.1 Mx here).

In [ ]:
YBCO_NN_A = 2.86          # projected nearest-neighbour cation-column distance (measured below)
TARGET_D = 10.0           # px between neighbouring columns after rescaling
YBCO_NORM_SIGMA = 2.5 * TARGET_D

def acf_neighbours(img, rmax=80):
    x = img - ndi.gaussian_filter(img, 30); x /= x.std()
    A = np.fft.fftshift(np.real(np.fft.ifft2(np.abs(np.fft.fft2(x)) ** 2))); A /= A.max()
    c = np.array(A.shape) // 2; sub = A[c[0] - rmax:c[0] + rmax + 1, c[1] - rmax:c[1] + rmax + 1]
    pk = peak_local_max(sub, min_distance=2, threshold_rel=0.05, exclude_border=False) - rmax
    r = np.hypot(*pk.T); keep = r > 2
    return np.sort(r[keep])[:4], sub

rows = []
for d in ybco:
    r, d["acf"] = acf_neighbours(d["img"])
    d["acf_nn_px"] = float(r[0])
    d["d_px"] = YBCO_NN_A / d["px_A"] if d["px_A"] else d["acf_nn_px"]
    d["scale"] = TARGET_D / d["d_px"]
    rows.append(dict(image=d["name"][:30], pixel_pm=round(d["px_A"] * 100, 2), acf_nn_px=round(r[0], 1),
                     acf_nn_A=round(r[0] * d["px_A"], 2), rescale_factor=round(d["scale"], 3)))
print(pd.DataFrame(rows).to_string())

fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
d = ybco[2]; w = int(40 / d["px_A"])
show(ax[0], d["img"][900:900 + w, 900:900 + w], f"YBCO {d['name'][:4]} (4 nm crop)")
ax[1].imshow(d["acf"], cmap="magma"); ax[1].axis("off"); ax[1].set_title("autocorrelation: lattice + column shape")
pr = d["img"][900:900 + w, 900:900 + w]; yy, xx = np.unravel_index(np.argmax(ndi.gaussian_filter(pr, 2)[10:-10, 10:-10]), (w - 20, w - 20))
yy += 10; xx += 10; rr = int(0.6 * d["d_px"])
ax[2].plot(np.arange(-rr, rr + 1) * d["px_A"], pr[yy, xx - rr:xx + rr + 1], "o-", label="horizontal")
ax[2].plot(np.arange(-rr, rr + 1) * d["px_A"], pr[yy - rr:yy + rr + 1, xx], "s-", label="vertical")
ax[2].set_xlabel("Å"); ax[2].set_title("intensity profile through one column"); ax[2].legend()
ax[3].hist([r["acf_nn_A"] for r in rows], bins=15); ax[3].set_xlabel("ACF nearest-neighbour distance (Å)")
ax[3].set_title("same spacing in every image -> scale prior")
plt.tight_layout(); plt.savefig(OUT / "ybco_geometry_exploration.png", dpi=110); plt.show()

## 2.2 Physics-based synthetic training data
Training images are rendered in the normalised frame (columns 10 px apart). The generator varies everything the network should *not* care about:
* lattice orientation and type (centred or primitive, angle 75–105°),
* row-dependent intensities (Ba/Y/Cu layering), missing rows and vacancies,
* column width, scan elongation, row-wise scan jitter,
* background, dark bands, and Poisson dose (2 to 200 counts at the column peak).

The label is a Gaussian heat-map (σ = 1.6 px) at every true column. The same scan distortion is applied to the label, so the target is where the column *appears*.

In [ ]:
def _aniso_kernels(sig, elong, ang, r=11, device="cpu"):
    yy, xx = torch.meshgrid(torch.arange(-r, r + 1, device=device, dtype=torch.float32),
                            torch.arange(-r, r + 1, device=device, dtype=torch.float32), indexing="ij")
    c, s = torch.cos(ang)[:, None, None], torch.sin(ang)[:, None, None]
    u = c * xx + s * yy; v = -s * xx + c * yy
    sa = (sig * elong)[:, None, None]; sb = sig[:, None, None]
    return torch.exp(-0.5 * ((u / sa) ** 2 + (v / sb) ** 2))[:, None]          # peak = 1

def _splat(canvas, b, x, y, w):
    B, Hc, Wc = canvas.shape
    x0 = torch.floor(x); y0 = torch.floor(y); fx = x - x0; fy = y - y0
    for dx, dy, ww in [(0, 0, (1 - fx) * (1 - fy)), (1, 0, fx * (1 - fy)), (0, 1, (1 - fx) * fy), (1, 1, fx * fy)]:
        xi = (x0 + dx).long(); yi = (y0 + dy).long()
        ok = (xi >= 0) & (xi < Wc) & (yi >= 0) & (yi < Hc) & (w > 0)
        canvas.view(-1).index_add_(0, (b * Hc * Wc + yi * Wc + xi)[ok], (ww * w)[ok])

@torch.no_grad()
def synth_atoms(B, P=96, device=DEVICE, g=None, dose_range=(2, 200)):
    """Return normalised images (B,1,P,P) and target heat-maps (B,1,P,P)."""
    pad = 32; Hc = P + 2 * pad
    U = lambda *s, lo=0., hi=1.: lo + (hi - lo) * torch.rand(*s, device=device, generator=g)
    th = U(B, hi=math.pi); L1 = U(B, lo=12.5, hi=16.5); L2 = U(B, lo=12.5, hi=16.5)
    gam = U(B, lo=math.radians(75), hi=math.radians(105))
    centred = U(B) < 0.75
    L1 = torch.where(centred, L1, L1 * 0.78); L2 = torch.where(centred, L2, L2 * 0.78)
    a1 = torch.stack([L1 * torch.cos(th), L1 * torch.sin(th)], -1)
    a2 = torch.stack([L2 * torch.cos(th + gam), L2 * torch.sin(th + gam)], -1)
    K = 9; ij = torch.arange(-K, K + 1, device=device, dtype=torch.float32)
    I_, J_ = torch.meshgrid(ij, ij, indexing="ij"); I_ = I_.reshape(-1); J_ = J_.reshape(-1)
    org = U(B, 2, lo=0, hi=1) * 15 + Hc / 2
    pos0 = org[:, None] + I_[None, :, None] * a1[:, None] + J_[None, :, None] * a2[:, None]
    pos1 = pos0 + 0.5 * (a1 + a2)[:, None]
    period = torch.randint(1, 4, (B,), device=device, generator=g)
    pat0 = U(B, 3, lo=0.35, hi=1.0); pat0[U(B, 3) < 0.12] = 0
    pat1 = U(B, 3, lo=0.3, hi=1.0); pat1[U(B, 3) < 0.08] = 0
    jm = (J_.long()[None] % period[:, None])
    I0 = torch.gather(pat0, 1, jm); I1 = torch.gather(pat1, 1, jm) * centred[:, None].float()
    pos = torch.cat([pos0, pos1], 1); inten = torch.cat([I0, I1], 1)
    inten = inten * (U(*inten.shape) > 0.02).float()                         # vacancies
    pos = pos + torch.randn(pos.shape, device=device, generator=g) * U(B, 1, 1, hi=0.35)
    bidx = torch.arange(B, device=device)[:, None].expand_as(inten)
    can = torch.zeros(B, Hc, Hc, device=device); lab = torch.zeros(B, Hc, Hc, device=device)
    _splat(can, bidx.reshape(-1), pos[..., 0].reshape(-1), pos[..., 1].reshape(-1), inten.reshape(-1))
    _splat(lab, bidx.reshape(-1), pos[..., 0].reshape(-1), pos[..., 1].reshape(-1), (inten > 0).float().reshape(-1))
    sig = U(B, lo=1.3, hi=2.6); el = torch.where(U(B) < 0.5, torch.ones(B, device=device), U(B, lo=1.0, hi=1.8))
    el = torch.minimum(el, 3.6 / sig)
    ker = _aniso_kernels(sig, el, U(B, hi=math.pi), device=device)
    img = F.conv2d(can[None], ker, padding=ker.shape[-1] // 2, groups=B)[0][:, None]
    lk = _aniso_kernels(torch.full((1,), 1.6, device=device), torch.ones(1, device=device), torch.zeros(1, device=device), r=6, device=device)
    heat = F.conv2d(lab[:, None], lk, padding=6).clamp(0, 1)
    # scan distortion: row-wise random-walk x-shift, applied to image AND label
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, Hc, device=device), torch.linspace(-1, 1, Hc, device=device), indexing="ij")
    rw = torch.cumsum(torch.randn(B, Hc, device=device, generator=g) * U(B, 1, hi=0.25), 1)
    rw = gblur(rw[:, None, None], 3.0)[:, 0, 0]; rw = rw - rw.mean(1, keepdim=True)
    grid = torch.stack([xx[None] + rw[:, :, None] * 2 / Hc, yy[None].expand(B, -1, -1)], -1)
    img = F.grid_sample(img, grid, align_corners=True, padding_mode="border")
    heat = F.grid_sample(heat, grid, align_corners=True, padding_mode="border")
    # background, dark bands along the layers, row gain jitter, noise
    bg = F.interpolate(U(B, 1, 5, 5), size=(Hc, Hc), mode="bicubic", align_corners=False)
    u = (-torch.sin(th))[:, None, None] * (xx[None] * Hc / 2) + torch.cos(th)[:, None, None] * (yy[None] * Hc / 2)
    band = 1 - U(B, 1, 1, hi=0.5) * (0.5 + 0.5 * torch.cos(2 * math.pi * u / U(B, 1, 1, lo=25, hi=90) + U(B, 1, 1, hi=6.3))) ** 4
    clean = (U(B, 1, 1, 1, lo=0.1, hi=1.2) * (1 + U(B, 1, 1, 1, hi=0.8) * (bg - 0.5)) + img) * band[:, None]
    clean = clean * (1 + torch.randn(B, 1, Hc, 1, device=device, generator=g) * U(B, 1, 1, 1, hi=0.04))
    dose = torch.exp(U(B, 1, 1, 1, lo=math.log(dose_range[0]), hi=math.log(dose_range[1]) + 1e-6))
    noisy = torch.poisson(clean.clamp(min=0) * dose, generator=g) / dose
    noisy = noisy + torch.randn(noisy.shape, device=device, generator=g) * U(B, 1, 1, 1, hi=0.05)
    x = local_norm(noisy, YBCO_NORM_SIGMA)
    s = slice(pad, pad + P)
    return x[..., s, s], heat[..., s, s]

xs_, hs_ = synth_atoms(8, P=128, g=torch.Generator(device=DEVICE).manual_seed(3))
fig, ax = plt.subplots(2, 8, figsize=(20, 5.4))
for i in range(8):
    show(ax[0, i], xs_[i, 0].cpu().numpy(), "synthetic"); ax[1, i].imshow(hs_[i, 0].cpu().numpy(), cmap="inferno"); ax[1, i].axis("off")
plt.tight_layout(); plt.show()

## 2.3 AtomUNet: a small object-specific network
A 2-level U-Net (16-32-48 channels, ~90 k parameters). The network only has to recognise *one* object, a 10-px-spaced column blob, because the geometry prior has removed scale. Its output is a heat-map; columns are its local maxima, refined to sub-pixel accuracy by a 5×5 heat-map centroid.

In [ ]:
def _block(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

class AtomUNet(nn.Module):
    def __init__(self, c=(16, 32, 48)):
        super().__init__()
        self.e1, self.e2, self.b = _block(1, c[0]), _block(c[0], c[1]), _block(c[1], c[2])
        self.u2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2); self.d2 = _block(2 * c[1], c[1])
        self.u1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2); self.d1 = _block(2 * c[0], c[0])
        self.out = nn.Conv2d(c[0], 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); b = self.b(F.max_pool2d(e2, 2))
        d2 = self.d2(torch.cat([self.u2(b), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

atom_net = AtomUNet().to(DEVICE)
print("AtomUNet trainable parameters:", sum(p.numel() for p in atom_net.parameters()))

def train_atoms(model, n_iter, B=16, lr=3e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n_iter)
    g = torch.Generator(device=DEVICE).manual_seed(42); hist = []
    model.train()
    for it in range(n_iter):
        x, h = synth_atoms(B, P=96, g=g)
        logit = model(x)
        loss = (F.binary_cross_entropy_with_logits(logit, h, reduction="none") * (1 + 4 * h)).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); hist.append(loss.item())
        if it % 200 == 0 or it == n_iter - 1:
            print(f"iter {it:4d}  loss {np.mean(hist[-100:]):.4f}")
    model.eval(); return hist

t = time.time()
atom_hist = train_atoms(atom_net, n_iter=700 if FAST else 2000)
print(f"YBCO training time: {time.time() - t:.1f}s")

## 2.4 Quantitative test on synthetic images with known coordinates
For a fair comparison, both the network and the classical difference-of-Gaussians (DoG) detector get their detection threshold tuned on a separate validation set. Matching uses the Hungarian algorithm with a tolerance of 3 px (0.3 × the column spacing, about 0.86 Å).

In [ ]:
def heat_peaks(h, thr, min_dist=4, win=2):
    pk = peak_local_max(h, min_distance=min_dist, threshold_abs=thr, exclude_border=False)
    if len(pk) == 0: return np.zeros((0, 2)), np.zeros(0)
    hp = np.pad(h, win, mode="constant")
    o = np.arange(-win, win + 1)
    Y = pk[:, 0, None, None] + win + o[None, :, None]; X = pk[:, 1, None, None] + win + o[None, None, :]
    w = np.clip(hp[Y, X], 0, None); s = w.sum((1, 2)) + 1e-9
    yy = pk[:, 0] + (w * o[None, :, None]).sum((1, 2)) / s; xx = pk[:, 1] + (w * o[None, None, :]).sum((1, 2)) / s
    return np.c_[yy, xx], h[pk[:, 0], pk[:, 1]]

def match(pred, gt, tol=3.0):
    if len(pred) == 0 or len(gt) == 0: return 0, len(pred), len(gt), np.zeros(0)
    D = cdist(pred, gt); r, c = linear_sum_assignment(D); ok = D[r, c] <= tol
    return int(ok.sum()), len(pred) - int(ok.sum()), len(gt) - int(ok.sum()), D[r, c][ok]

def dog(x):
    return (gblur(x, 1.5) - gblur(x, 4.0))

def make_eval_set(n, seed, P=128):
    g = torch.Generator(device=DEVICE).manual_seed(seed); xs, hs = [], []
    for _ in range(n // 16):
        x, h = synth_atoms(16, P=P, g=g); xs.append(x.cpu()); hs.append(h.cpu())
    X = torch.cat(xs); Hh = torch.cat(hs)
    gts = [heat_peaks(Hh[i, 0].numpy(), 0.5)[0] for i in range(len(X))]
    return X, gts

@torch.no_grad()
def maps_for(X):
    nn_h = torch.cat([torch.sigmoid(atom_net(X[i:i + 32].to(DEVICE))).cpu() for i in range(0, len(X), 32)])
    dg = dog(X); dg = dg / dg.flatten(1).std(1)[:, None, None, None]
    return nn_h[:, 0].numpy(), dg[:, 0].numpy()

def score(maps, gts, thr, border=4):
    TP = FP = FN = 0; errs = []
    for m, gt in zip(maps, gts):
        p, _ = heat_peaks(m, thr); P_ = m.shape[0]
        inb = lambda a: a[(a[:, 0] >= border) & (a[:, 0] < P_ - border) & (a[:, 1] >= border) & (a[:, 1] < P_ - border)] if len(a) else a
        tp, fp, fn, e = match(inb(p), inb(gt)); TP += tp; FP += fp; FN += fn; errs.append(e)
    e = np.concatenate(errs) if errs else np.zeros(0)
    prec = TP / max(TP + FP, 1); rec = TP / max(TP + FN, 1)
    return dict(precision=prec, recall=rec, F1=2 * prec * rec / max(prec + rec, 1e-9),
                rmse_px=float(np.sqrt(np.mean(e ** 2))) if len(e) else np.nan)

Xv, gv = make_eval_set(64, seed=7)
nn_v, dog_v = maps_for(Xv)
thr_nn = max(np.arange(0.1, 0.9, 0.05), key=lambda t: score(nn_v, gv, t)["F1"])
thr_dog = max(np.arange(0.2, 4.0, 0.1), key=lambda t: score(dog_v, gv, t)["F1"])
print(f"tuned thresholds: AtomUNet {thr_nn:.2f} | DoG {thr_dog:.2f}")

Xt, gt_t = make_eval_set(192, seed=99)
nn_t, dog_t = maps_for(Xt)
res = []
for name, maps, thr in [("DoG + peak finding (classical)", dog_t, thr_dog), ("AtomUNet (ours)", nn_t, thr_nn)]:
    r_ = score(maps, gt_t, thr); r_["rmse_A"] = r_["rmse_px"] * YBCO_NN_A / TARGET_D; res.append(dict(method=name, **r_))
syn_ybco = pd.DataFrame(res).round(4); print(syn_ybco.to_string(index=False)); syn_ybco.to_csv(OUT / "ybco_synthetic_benchmark.csv", index=False)

In [ ]:
# Robustness versus electron dose (counts at the column peak)
rows = []
for dose in [2, 4, 8, 16, 32, 64, 128]:
    g = torch.Generator(device=DEVICE).manual_seed(1000 + dose); xs, hs = [], []
    for _ in range(4):
        x, h = synth_atoms(16, P=128, g=g, dose_range=(dose, dose)); xs.append(x.cpu()); hs.append(h.cpu())
    X = torch.cat(xs); gts = [heat_peaks(hh[0].numpy(), 0.5)[0] for hh in torch.cat(hs)]
    a, b = maps_for(X); sa, sb = score(a, gts, thr_nn), score(b, gts, thr_dog)
    rows.append(dict(dose=dose, F1_AtomUNet=sa["F1"], F1_DoG=sb["F1"],
                     rmse_A_AtomUNet=sa["rmse_px"] * YBCO_NN_A / TARGET_D, rmse_A_DoG=sb["rmse_px"] * YBCO_NN_A / TARGET_D))
dose_df = pd.DataFrame(rows).round(4); print(dose_df.to_string(index=False)); dose_df.to_csv(OUT / "ybco_dose_robustness.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].semilogx(dose_df.dose, dose_df.F1_AtomUNet, "o-", label="AtomUNet"); ax[0].semilogx(dose_df.dose, dose_df.F1_DoG, "s--", label="DoG")
ax[0].set_xlabel("counts at column peak (dose)"); ax[0].set_ylabel("F1"); ax[0].legend(); ax[0].set_title("Detection F1 vs dose")
ax[1].semilogx(dose_df.dose, dose_df.rmse_A_AtomUNet, "o-", label="AtomUNet"); ax[1].semilogx(dose_df.dose, dose_df.rmse_A_DoG, "s--", label="DoG")
ax[1].set_xlabel("counts at column peak (dose)"); ax[1].set_ylabel("localisation RMSE (Å)"); ax[1].legend(); ax[1].set_title("Localisation error vs dose")
i = 5
p_nn, _ = heat_peaks(nn_t[i], thr_nn); p_dg, _ = heat_peaks(dog_t[i], thr_dog)
show(ax[2], Xt[i, 0].numpy(), "synthetic test: o truth, + AtomUNet, x DoG")
ax[2].plot(gt_t[i][:, 1], gt_t[i][:, 0], "o", mfc="none", mec="lime", ms=9)
ax[2].plot(p_nn[:, 1], p_nn[:, 0], "r+", ms=8); ax[2].plot(p_dg[:, 1], p_dg[:, 0], "cx", ms=6)
plt.tight_layout(); plt.savefig(OUT / "ybco_synthetic_benchmark.png", dpi=110); plt.show()

## 2.5 Apply to all real YBCO images
Steps: rescale so columns are 10 px apart → local normalisation → tiled AtomUNet → sub-pixel peaks → map back to original pixel coordinates. Each column also gets an intensity, used to split columns into **bright** (likely Ba/Y) and **dim** (likely Cu–O) groups.

**Real-data quality control (no ground truth):** the nearest-neighbour distance in Å must be physical (≈2.8 Å, or ≈3.9–4.1 Å if the centred columns are invisible), and the **lattice regularity** (fraction of columns whose NN distance is within ±15 % of the median) should be close to 1.

In [ ]:
@torch.no_grad()
def predict_tiled(model, x, tile=512, ov=32):
    model.eval(); _, _, H, W = x.shape; step = tile - 2 * ov
    Hp, Wp = math.ceil(H / step) * step, math.ceil(W / step) * step
    xp = F.pad(x, (ov, ov + Wp - W, ov, ov + Hp - H), mode="reflect")
    out = torch.zeros(1, 1, Hp, Wp)
    for y in range(0, Hp, step):
        for x0 in range(0, Wp, step):
            o = torch.sigmoid(model(xp[..., y:y + tile, x0:x0 + tile].to(DEVICE))).cpu()
            out[..., y:y + step, x0:x0 + step] = o[..., ov:ov + step, ov:ov + step]
    return out[..., :H, :W]

def find_columns(d, method="nn"):
    img = d["img"]; H, W = img.shape; s = d["scale"]
    Hr, Wr = int(round(H * s)), int(round(W * s))
    x = F.interpolate(t2(img), size=(Hr, Wr), mode="bilinear", align_corners=False, antialias=s < 1)
    x = local_norm(x.to(DEVICE), YBCO_NORM_SIGMA)
    if method == "nn":
        h = predict_tiled(atom_net, x)[0, 0].numpy(); thr = thr_nn
    else:
        h = dog(x); h = (h / h.std())[0, 0].cpu().numpy(); thr = thr_dog
    pts, sc = heat_peaks(h, thr)
    pts = (pts + 0.5) * np.array([H / Hr, W / Wr]) - 0.5
    return pts, sc, h

def lattice_qc(pts, px_A):
    if len(pts) < 3: return np.nan, np.nan, np.zeros(0)
    dnn = cKDTree(pts).query(pts, k=2)[0][:, 1] * px_A
    med = np.median(dnn)
    return med, float(np.mean(np.abs(dnn / med - 1) < 0.15)), dnn

(OUT / "ybco_atoms").mkdir(exist_ok=True)
rows = []; t = time.time()
for d in ybco:
    pts, sc, _ = find_columns(d, "nn")
    ib = d["img"] - ndi.gaussian_filter(d["img"], 2 * d["d_px"])
    ib = ndi.gaussian_filter(ib, 0.2 * d["d_px"])
    inten = ndi.map_coordinates(ib, pts.T, order=1) if len(pts) else np.zeros(0)
    cls = (inten > threshold_otsu(inten)) if len(pts) > 10 else np.ones(len(pts), bool)
    d.update(pts=pts, score=sc, inten=inten, bright=cls)
    med, reg, dnn = lattice_qc(pts, d["px_A"]); d["dnn"] = dnn
    pts_d, _, _ = find_columns(d, "dog"); med_d, reg_d, _ = lattice_qc(pts_d, d["px_A"]); d["pts_dog"] = pts_d
    area_nm2 = d["img"].size * (d["px_A"] / 10) ** 2
    rows.append(dict(image=d["name"][:28], n_columns=len(pts), columns_per_nm2=round(len(pts) / area_nm2, 2),
                     median_nn_A=round(med, 3), regularity=round(reg, 3),
                     n_DoG=len(pts_d), median_nn_A_DoG=round(med_d, 3), regularity_DoG=round(reg_d, 3)))
    pd.DataFrame(dict(y_px=pts[:, 0], x_px=pts[:, 1], y_nm=pts[:, 0] * d["px_A"] / 10, x_nm=pts[:, 1] * d["px_A"] / 10,
                      nn_score=sc, column_intensity=inten, bright_column=cls)).to_csv(
        OUT / "ybco_atoms" / f"{d['name'].replace(' ', '_')}_atoms.csv", index=False)
ybco_summary = pd.DataFrame(rows); print(ybco_summary.to_string()); ybco_summary.to_csv(OUT / "ybco_summary.csv", index=False)
print(f"inference on {len(ybco)} images: {time.time() - t:.1f}s")
print("mean regularity  AtomUNet: %.3f | DoG: %.3f" % (ybco_summary.regularity.mean(), ybco_summary.regularity_DoG.mean()))

In [ ]:
fig, ax = plt.subplots(3, 6, figsize=(21, 11)); ax = ax.ravel()
for i, d in enumerate(ybco):
    w = int(50 / d["px_A"]); y0 = x0 = 1024 - w // 2
    show(ax[i], d["img"][y0:y0 + w, x0:x0 + w], f"{d['name'][:4]}  ({d['px_A'] * 100:.0f} pm/px, 5 nm)")
    p = d["pts"] - [y0, x0]; m = (p[:, 0] >= 0) & (p[:, 0] < w) & (p[:, 1] >= 0) & (p[:, 1] < w)
    ms = max(2.0, 6 * 60 / w)
    ax[i].plot(p[m & d["bright"], 1], p[m & d["bright"], 0], "o", mfc="none", mec="red", ms=ms, mew=1)
    ax[i].plot(p[m & ~d["bright"], 1], p[m & ~d["bright"], 0], "o", mfc="none", mec="cyan", ms=ms, mew=1)
plt.suptitle("Challenge 2 – AtomUNet atomic columns (red = bright, likely Ba/Y; cyan = dim, likely Cu-O)", fontsize=14)
plt.tight_layout(); plt.savefig(OUT / "ybco_atoms_all_images.png", dpi=100); plt.show()

In [ ]:
d = ybco[2]
fig, ax = plt.subplots(1, 3, figsize=(19, 6.4))
show(ax[0], d["img"], f"{d['name'][:4]}: {len(d['pts'])} columns (full frame)")
ax[0].plot(d["pts"][:, 1], d["pts"][:, 0], "r.", ms=0.8)
w = int(30 / d["px_A"]); y0, x0 = 300, 300
show(ax[1], d["img"][y0:y0 + w, x0:x0 + w], "3 nm zoom: AtomUNet (red o) vs DoG (cyan x)")
p = d["pts"] - [y0, x0]; m = (p >= 0).all(1) & (p < w).all(1); ax[1].plot(p[m, 1], p[m, 0], "o", mfc="none", mec="r", ms=10)
q = d["pts_dog"] - [y0, x0]; m = (q >= 0).all(1) & (q < w).all(1); ax[1].plot(q[m, 1], q[m, 0], "cx", ms=7)
allnn = np.concatenate([dd["dnn"] for dd in ybco])
ax[2].hist(allnn, bins=np.linspace(1.5, 5.5, 120), color="C0")
for v, lab in [(2.86, "2.86 Å (centred-square)"), (4.05, "~4 Å (a or c/3)")]:
    ax[2].axvline(v, color="r", ls="--"); ax[2].text(v + 0.05, ax[2].get_ylim()[1] * 0.9, lab, color="r", fontsize=8)
ax[2].set_xlabel("nearest-neighbour column distance (Å)"); ax[2].set_title("All 18 images: NN distances of detected columns")
plt.tight_layout(); plt.savefig(OUT / "ybco_full_frame_and_nn_hist.png", dpi=110); plt.show()

---
## 3. Summary of results

In [ ]:
print("=== Challenge 1: SiSb boundary (synthetic test with exact ground truth) ===")
print(syn_sisb.to_string())
print("\n=== Challenge 2: YBCO atom finding (synthetic test with exact ground truth) ===")
print(syn_ybco.to_string(index=False))
print("\n=== Model sizes ===")
print("BraggShapeNet:", sum(p.numel() for p in sisb_net.parameters()), "parameters")
print("AtomUNet     :", sum(p.numel() for p in atom_net.parameters()), "parameters")
torch.save(sisb_net.state_dict(), OUT / "braggshapenet_sisb.pt")
torch.save(atom_net.state_dict(), OUT / "atomunet_ybco.pt")
print(f"\nTotal runtime: {(time.time() - T0) / 60:.1f} min on {DEVICE}. All outputs in {OUT.resolve()}")

# Uncomment in Colab to download everything as a zip:
# import shutil; shutil.make_archive("day25_results", "zip", OUT); from google.colab import files; files.download("day25_results.zip")

---
## 4. Discussion: what works, what does not, limitations

**Connecting object geometry to network design**

| | SiSb boundary | YBCO columns |
|---|---|---|
| Object | a *texture region* (periodic vs. disordered) | a *compact blob* on a lattice |
| Key geometric quantity | lattice radii \|k\| from the global FFT (Si 111 / 220) | nearest-neighbour column spacing (≈2.86 Å, from ACF + pixel size) |
| How it enters the network | fixed-position, learnable-width Fourier filter bank + max/mean over angle, which makes it rotation-invariant by construction | input rescaled so spacing = 10 px; atom size and spacing are then constants |
| Trainable parameters | ~8 k | ~95 k |
| Training data | synthetic (exact masks) + confident pseudo-labels | synthetic only (exact coordinates) |

**What works**
* **SiSb:** HAADF intensity is the wrong feature. The dark loop is not the phase boundary, and on the synthetic test an intensity threshold reaches mIoU 0.63, the classical order map 0.77 and BraggShapeNet **0.83**, with a boundary error of about 0.8 lattice periods. The rotation-invariant Bragg feature isolates crystal from amorphous. It also catches the **second, twinned grain** below the loop, which a detector tuned to one orientation misses. On held-out real frames the network agrees with the classical map to within less than one lattice spacing. As a by-product, the frames read out the crystal/amorphous front evolving over time (area fraction and boundary length per frame).
* **YBCO:** after scale normalisation one tiny network handles 1000 kx – 4.1 Mx images. On synthetic ground truth it beats a threshold-tuned DoG detector at **every dose** (F1 0.96 vs 0.91 overall; localisation 0.18 Å vs 0.24 Å RMSE). On the 18 real images, the detected column density (≈12.3 per nm²) matches the expected 2 columns per 4.05² Å² cell (12.2 per nm²). Lattice regularity is 0.97 for AtomUNet vs 0.75 for DoG.

**What does not work / limitations**
* There is no human-annotated ground truth for the real images. Real-data accuracy is judged by self-consistency: agreement with the physics baseline, physical NN distances, column density and lattice regularity.
* **SiSb:** the boundary is resolved only to about 1 lattice period (the order parameter needs several periods of fringes). Very thin crystalline slivers, or regions where the lattice is tilted off the zone axis, look amorphous. The Otsu-based pseudo-labels inherit the classical map's errors; the ignore band and the synthetic data reduce, but do not remove, this bias.
* **YBCO:** the most extreme low-dose image (1115) and images with strong scan distortion (1709, 1717, 1333/1338) give lower regularity (0.88–0.92). The bright/dim column classification is a simple intensity threshold, not a chemically validated assignment. The 2.86 Å prior is YBCO-specific; for other materials it should be replaced by the ACF measurement, which the code already falls back to when no pixel size is available.
* **Next steps:** self-training on real YBCO images with lattice-consistent pseudo-labels, and an atom-class head (Ba/Y/Cu) trained on simulated HAADF (e.g. abTEM). For SiSb, tracking boundary velocity between frames to measure crystallisation kinetics.